# D1-04 First LCA from demand to score in Brightway

This notebook includes the course database setup by importing BAFU:2025 from Excel.


## Learning goals

- Initialize a Brightway project for the course.
- Import the BAFU:2025 Brightway-compatible Excel inventory.
- Run a first LCA from a demand to an impact score.


## 1) Set project and ensure core setup

`bw2setup()` creates core biosphere and LCIA method data used by Brightway.


In [ ]:
import bw2data as bd
import bw2io as bi

project_name = 'barcelona-rlcia-2026'
bd.projects.set_current(project_name)
print('Current project:', bd.projects.current)

if 'biosphere3' not in bd.databases:
    bi.bw2setup()
    print('Created core biosphere and methods.')
else:
    print('Core biosphere/methods already available.')

print('Databases:', list(bd.databases))
print('Number of methods:', len(bd.methods))


## 2) Check BAFU file location

Expected file location:
- `tutorials/DAY 1/assets/BAFU_2025_inventory.xlsx`


In [ ]:
from pathlib import Path

bafu_file = Path('tutorials/DAY 1/assets/BAFU_2025_inventory.xlsx')
print('BAFU file path:', bafu_file.resolve())
print('Found:', bafu_file.exists())


## 3) Import BAFU:2025 database

This imports the Excel file into Brightway as a database named `bafu-2025`.


In [ ]:
database_name = 'bafu-2025'

if not bafu_file.exists():
    print('BAFU file not found. Place it in tutorials/DAY 1/assets/.')
else:
    if database_name in bd.databases:
        print(f"Database '{database_name}' already exists. Skipping import.")
    else:
        importer = bi.ExcelImporter(str(bafu_file))
        importer.apply_strategies()

        if 'biosphere3' in bd.databases:
            importer.match_database('biosphere3', fields=('name', 'unit', 'categories'))

        importer.statistics()
        importer.write_database(database_name)
        print(f"Imported database: {database_name}")

print('Databases now:', list(bd.databases))


## 4) Run your first LCA on the imported database

The cell below picks one activity from `bafu-2025` and one available LCIA method,
then calculates an LCA score.


In [ ]:
import bw2calc as bc

if 'bafu-2025' not in bd.databases:
    print("'bafu-2025' is not available yet. Run the import cell first.")
else:
    db = bd.Database('bafu-2025')
    method = next(iter(bd.methods))
    activity = db.random()

    print('Selected activity:', activity['name'])
    print('Method:', method)

    lca = bc.LCA({activity: 1}, method)
    lca.lci()
    lca.lcia()
    print('LCA score:', lca.score)


## Exercise

1. Import the same BAFU file under a second name: `bafu-2025-training`.
2. Run an LCA score for a random activity from `bafu-2025-training`.


In [ ]:
# TODO


In [ ]:
db_name = 'bafu-2025-training'

if not bafu_file.exists():
    print('BAFU file not found.')
else:
    if db_name not in bd.databases:
        imp = bi.ExcelImporter(str(bafu_file))
        imp.apply_strategies()
        if 'biosphere3' in bd.databases:
            imp.match_database('biosphere3', fields=('name', 'unit', 'categories'))
        imp.write_database(db_name)

    db = bd.Database(db_name)
    method = next(iter(bd.methods))
    act = db.random()
    lca = bc.LCA({act: 1}, method)
    lca.lci()
    lca.lcia()
    print('Activity:', act['name'])
    print('Score:', lca.score)
